# Module 06 — Assignment: backpropagation through time

Module 05's assignment built what a very deep feed-forward network needed. This one builds a
network that is deep in *time*: one layer applied again at every step of a sequence. You
implement four load-bearing pieces — **one step of the recurrence**, the **forward pass over a
whole sequence**, **backpropagation through time** (with Elman's truncation as an option), and
**gradient-norm clipping** — and the check cells grade you against finite differences and the
canonical `python/rnn.py` + `nanograd` answer key. The finale drops your recurrent layer into a
GIVEN harness that must reproduce the module's echo result: with full backpropagation through
time your network learns to output the symbol it read four steps earlier, perfectly; with the
gradient cut after one step, it cannot.

Rules: don't peek at `lib/python/nanograd` or `python/rnn.py` until you've tried. The check cells
are your feedback. Fill every `# TODO`, run top to bottom.

In [ ]:
# Setup — run once. (GIVEN: do not modify.)
import os, sys, math
import numpy as np

def _find_repo():
    here = os.getcwd()
    cands = [here] + [os.path.abspath(os.path.join(here, *(['..'] * k))) for k in range(1, 6)]
    for base in cands:
        if os.path.isdir(os.path.join(base, 'lib', 'python', 'nanograd')):
            return base
    return here

REPO = _find_repo()
TOPIC = os.path.join(REPO, 'topics', '06-rnn')
sys.path.insert(0, os.path.join(REPO, 'lib', 'python'))
sys.path.insert(0, os.path.join(TOPIC, 'python'))
sys.path.insert(0, os.path.join(TOPIC, 'tests'))
import nanograd as ng
import rnn as canon
from check_utils import (rel_error, eval_numerical_gradient_array, compare_to_canonical)
from nanograd.rng import Rng
print('setup ok — nanograd and check utils imported')

## Part 1 — one step of the recurrence

A recurrent layer keeps a hidden state $h$ and updates it from each new input with the *same*
weights every time:

$$h^{\langle t\rangle} = \tanh\big(x^{\langle t\rangle} W_{xh} + h^{\langle t-1\rangle} W_{hh} + b_h\big).$$

Shapes are row-major with rows = samples, as everywhere in this book: $x$ is $(m, n_{in})$,
$h$ is $(m, n_h)$, $W_{xh}$ is $(n_{in}, n_h)$, $W_{hh}$ is $(n_h, n_h)$. Implement one step.

In [ ]:
def rnn_step(x, h_prev, Wxh, Whh, bh):
    h = None
    ###########################################################################
    # TODO: one step of the recurrence: the new state from the input and the
    # previous state. One line.
    ###########################################################################

    ###########################################################################
    #                          END OF YOUR CODE
    ###########################################################################
    return h

In [ ]:
# Check: one step equals nanograd's RNN run for a single step from the same state.
rng = np.random.default_rng(0)
lib = ng.RNN(4, 6)
for p in lib.params():
    p[...] = rng.standard_normal(p.shape) * 0.5
x, h_prev = rng.standard_normal((3, 4)), rng.standard_normal((3, 6)) * 0.5
yours = rnn_step(x, h_prev, lib.Wxh, lib.Whh, lib.bh)
theirs = lib.forward(x[None], h_prev)[0]
assert yours is not None and yours.shape == (3, 6), 'rnn_step should return an (m, n_h) array'
print(f'rel_error vs nanograd = {rel_error(yours, theirs):.1e}'); assert rel_error(yours, theirs) < 1e-12
compare_to_canonical([float(yours.sum()), float((yours ** 2).sum())],
                     [float(theirs.sum()), float((theirs ** 2).sum())], labels=['h_sum', 'h_sq'])

### Inline Question 1
(a) The module's Shakespeare model reads one-hot characters (65 of them) into 256 hidden units.
How many numbers are in $W_{xh}$, $W_{hh}$ and $b_h$ — and how does that change if the text is
ten times longer? How many weights would a *feed-forward* layer need to read the last 100
characters at once into 256 units? (b) $x$ is one-hot. What does $x W_{xh}$ actually compute,
and why do frameworks implement it as a table lookup (an "embedding") instead of a matrix
product? *Your answer:* ____

## Part 2 — the forward pass over a sequence

Inputs come **time-major**: $X$ has shape $(T, m, n_{in})$ — $T$ steps of an $(m, n_{in})$ batch.
Starting from $h^{\langle -1\rangle} = h_0$ (zeros if none is given), apply `rnn_step` at every step
and keep **every** state: $H$ has shape $(T, m, n_h)$. Store it as `self.H` — backpropagation needs
all of them.

In [ ]:
class MyRNN:
    def __init__(self, n_in, n_h):
        self.Wxh, self.Whh, self.bh = np.zeros((n_in, n_h)), np.zeros((n_h, n_h)), np.zeros(n_h)  # GIVEN
        self.dWxh, self.dWhh, self.dbh = np.zeros_like(self.Wxh), np.zeros_like(self.Whh), np.zeros_like(self.bh)

    def forward(self, X, h0=None):
        T, m, _ = X.shape                                          # GIVEN
        n_h = self.Whh.shape[0]
        self.X = X
        self.h0 = np.zeros((m, n_h)) if h0 is None else h0
        H = None
        ###########################################################################
        # TODO: run rnn_step over t = 0 .. T-1, starting from self.h0; keep every
        # state in H, shape (T, m, n_h); remember it as self.H. ~5 lines.
        ###########################################################################

        ###########################################################################
        #                          END OF YOUR CODE
        ###########################################################################
        return H

    def backward(self, dH, truncate=0):
        return my_bptt(self, dH, truncate)                         # GIVEN: Part 3

    def params(self): return [self.Wxh, self.Whh, self.bh]
    def grads(self): return [self.dWxh, self.dWhh, self.dbh]

In [ ]:
# Check: identical to nanograd's RNN on a sequence, with and without an initial state.
rng = np.random.default_rng(1)
you, lib = MyRNN(4, 6), ng.RNN(4, 6)
for p, q in zip(you.params(), lib.params()):
    p[...] = q[...] = rng.standard_normal(p.shape) * 0.5
X, h0 = rng.standard_normal((9, 3, 4)), rng.standard_normal((3, 6)) * 0.5
for start in (None, h0):
    Hy, Hl = you.forward(X, start), lib.forward(X, start)
    assert Hy is not None and Hy.shape == (9, 3, 6), 'forward should return (T, m, n_h)'
    assert rel_error(Hy, Hl) < 1e-12, 'forward differs from nanograd'
print('every state matches nanograd; the last state:', np.round(Hy[-1, 0], 4))
compare_to_canonical([float(Hy.sum()), float(Hy[-1].sum())], [float(Hl.sum()), float(Hl[-1].sum())],
                     labels=['H_sum', 'h_T'])

### Inline Question 2
A feed-forward layer processes a whole batch with one matrix product. Your forward pass is a
`for` loop over time. Which part of the computation could be done for all $T$ steps at once
(nanograd's `RNN.forward` does it), and why can't the rest? What does that mean for how fast a
recurrent network can be trained on hardware that thrives on large matrix products — and what
will Module 13's Transformer do about it? *Your answer:* ____

## Part 3 — backpropagation through time

Given $dH$ — the gradient $\partial L/\partial h^{\langle t\rangle}$ arriving from whatever read each step's
state — walk time **backwards**. At every step the state's gradient has two sources, the output at
$t$ and the future through $h^{\langle t+1\rangle}$; the shared weights collect a share from every step:

$$dh^{\langle t\rangle} = dH^{\langle t\rangle} + dz^{\langle t+1\rangle} W_{hh}^{\top}, \qquad dz^{\langle t\rangle} = dh^{\langle t\rangle} \odot \big(1 - (h^{\langle t\rangle})^2\big),$$

$$dW_{xh} \mathrel{+}= x^{\langle t\rangle\top} dz^{\langle t\rangle},\qquad dW_{hh} \mathrel{+}= h^{\langle t-1\rangle\top} dz^{\langle t\rangle},\qquad db_h \mathrel{+}= \textstyle\sum_{\text{rows}} dz^{\langle t\rangle},\qquad dX^{\langle t\rangle} = dz^{\langle t\rangle} W_{xh}^{\top},$$

with $h^{\langle -1\rangle} = h_0$, and $\partial L/\partial h_0 = dz^{\langle 0\rangle} W_{hh}^{\top}$.

**Truncation.** With `truncate = k > 0`, the gradient is not passed from step $t$ back to step
$t-1$ (or to $h_0$) whenever $t \bmod k = 0$ — the path into the past is cut there, exactly as if
the sequence had been trained in chunks of $k$ steps. `truncate=1` is Elman's 1990 training.

Set `rnn.dWxh`, `rnn.dWhh`, `rnn.dbh` and `rnn.dh0`; return $dX$.

In [ ]:
def my_bptt(rnn, dH, truncate=0):
    T = dH.shape[0]
    dX = None
    ###########################################################################
    # TODO: zero the three weight gradients; walk t = T-1 .. 0 carrying the
    # gradient from the future (dh_next); at each step compute dz, add each
    # weight's share, fill dX[t], and pass dz Whh^T one step back -- unless
    # truncation cuts it. Store the gradients on rnn (dh0 = what is passed
    # back from step 0). ~13 lines.
    ###########################################################################

    ###########################################################################
    #                          END OF YOUR CODE
    ###########################################################################
    return dX

In [ ]:
# Check 1: finite differences, every input and parameter, with an initial state.
rng = np.random.default_rng(2)
you = MyRNN(3, 5)
for p in you.params():
    p[...] = rng.standard_normal(p.shape) * 0.6
X, h0, dH = rng.standard_normal((7, 2, 3)), rng.standard_normal((2, 5)) * 0.5, rng.standard_normal((7, 2, 5))
f = lambda _: you.forward(X, h0)
f(None); dX = you.backward(dH)
assert dX is not None, 'my_bptt should return dX'
err = rel_error(dX, eval_numerical_gradient_array(f, X, dH)); print(f'dX   rel_error = {err:.1e}'); assert err < 1e-6
f(None); you.backward(dH); dh0 = you.dh0
err = rel_error(dh0, eval_numerical_gradient_array(f, h0, dH)); print(f'dh0  rel_error = {err:.1e}'); assert err < 1e-6
for name in ('Wxh', 'Whh', 'bh'):
    f(None); you.backward(dH); g = getattr(you, 'd' + name).copy()
    err = rel_error(g, eval_numerical_gradient_array(f, getattr(you, name), dH))
    print(f'd{name:3s} rel_error = {err:.1e}'); assert err < 1e-6

# Check 2: truncation is chunked training -- the truncated gradient equals the sum of the
# chunks' full gradients, each chunk starting from the state the previous one left.
for k in (1, 2, 3):
    H = you.forward(X, h0); you.backward(dH, truncate=k); got = [g.copy() for g in you.grads()]
    want = [np.zeros_like(g) for g in got]
    piece = MyRNN(3, 5); piece.Wxh, piece.Whh, piece.bh = you.Wxh, you.Whh, you.bh
    for s in range(0, 7, k):
        piece.forward(X[s:s + k], h0 if s == 0 else H[s - 1]); piece.backward(dH[s:s + k])
        want = [w + g for w, g in zip(want, piece.grads())]
    err = max(rel_error(a, b) for a, b in zip(got, want))
    print(f'truncate={k}: vs chunked BPTT rel_error = {err:.1e}, dh0 = 0: {bool(np.all(you.dh0 == 0))}')
    assert err < 1e-12 and np.all(you.dh0 == 0)

lib = ng.RNN(3, 5); lib.Wxh[:], lib.Whh[:], lib.bh[:] = you.Wxh, you.Whh, you.bh
you.forward(X, h0); dXy = you.backward(dH, truncate=2); lib.forward(X, h0); dXl = lib.backward(dH, truncate=2)
compare_to_canonical([float(dXy.sum()), float(you.dWhh.sum()), float(you.dbh.sum())],
                     [float(dXl.sum()), float(lib.dWhh.sum()), float(lib.dbh.sum())],
                     labels=['dX', 'dWhh', 'dbh'])

### Inline Question 3
Unroll your loop: $\partial L/\partial h^{\langle t-k\rangle}$ is $\partial L/\partial h^{\langle t\rangle}$ times a
product of $k$ matrices. Write it. Suppose the units stay in tanh's linear region
($1 - h^2 \approx 1$) and $W_{hh} = 0.5\,I$: by what factor does a gradient shrink over 20 steps?
With $W_{hh} = 1.1\,I$? Why can't you simply set $W_{hh} = I$ and be done — what happens to the
forward pass, and to the tanh derivatives, once the state has to *change*? *Your answer:* ____

## Part 4 — gradient-norm clipping

Pascanu, Mikolov and Bengio (2013): treat all the gradients as one long vector $g$; if
$\|g\| > \tau$, rescale it to length $\tau$ — same direction, shorter step:

$$g \leftarrow g \cdot \frac{\tau}{\|g\|} \quad\text{if } \|g\| > \tau .$$

Scale the arrays **in place** (the optimizer reads them afterwards) and return the norm measured
*before* clipping.

In [ ]:
def my_clip_grad_norm(grads, max_norm):
    norm = None
    ###########################################################################
    # TODO: the norm of all the gradients together; if it exceeds max_norm,
    # scale every array in place by max_norm / norm. ~4 lines.
    ###########################################################################

    ###########################################################################
    #                          END OF YOUR CODE
    ###########################################################################
    return norm

In [ ]:
# Check: the same norm and the same clipped gradients as nanograd's, direction kept,
# and a no-op below the threshold.
rng = np.random.default_rng(3)
gs = [rng.standard_normal((4, 5)) * 2, rng.standard_normal(6) * 2]
ref = [g.copy() for g in gs]
n_you = my_clip_grad_norm(gs, 1.5); n_lib = ng.clip_grad_norm(ref, 1.5)
assert n_you is not None, 'return the norm'
after = np.sqrt(sum(float((g * g).sum()) for g in gs))
print(f'norm before {n_you:.6f} (nanograd {n_lib:.6f}); after clipping {after:.6f}')
assert abs(n_you - n_lib) < 1e-12 and abs(after - 1.5) < 1e-12
assert max(rel_error(a, b) for a, b in zip(gs, ref)) < 1e-12
same = [g.copy() for g in gs]; my_clip_grad_norm(gs, 10.0)
assert all(np.array_equal(a, b) for a, b in zip(gs, same)), 'below max_norm, nothing may change'
compare_to_canonical([n_you, float(gs[0].sum())], [n_lib, float(ref[0].sum())], labels=['norm', 'g_sum'])

### Inline Question 4
Karpathy's 2015 `min-char-rnn.py` clipped every gradient *entry* to $[-5, 5]$ instead. Give a
two-component gradient where the two rules produce steps in different directions. Which rule
preserves the direction of steepest descent? And why do the module's Adam runs never trigger the
clip at all? (Hint: what does Adam divide each step by?) *Your answer:* ____

## Verify: your pieces reproduce the module's echo result

A GIVEN harness trains the echo network — **your** `MyRNN` reading the 4-symbol stream, a
library `Linear` read-out, Adam, and **your** clipping — with exactly the initial weights and
batches of `canon.train_echo`, for 300 updates. With full backpropagation through time your
network must echo the symbol from four steps back perfectly on 96-step test sequences; with
Elman's one-step truncation it must fail. Both must match the canonical runs.

In [ ]:
def train_echo_with(rnn_cls, clip_fn, truncate, steps=canon.TOY_STEPS, seed=0):
    '''GIVEN harness: canon.train_echo with YOUR layer and YOUR clipping. Do not modify.'''
    gen = np.random.default_rng(canon.DATA_SEED + 100 * seed)
    r = Rng(canon.TOY_SEED + 100 * seed)
    rnn = rnn_cls(canon.V, canon.N_H); ng.xavier_normal(rnn.Wxh, r); ng.xavier_normal(rnn.Whh, r)
    head = ng.Linear(canon.N_H, canon.V); ng.xavier_normal(head.W, r)
    opt, loss_layer, first = ng.Adam(lr=canon.TOY_LR), ng.SoftmaxCrossEntropy(), None
    for k in range(steps):
        S = gen.integers(0, canon.V, size=(canon.T_TOY, canon.M_TOY))
        loss = canon.echo_loss_and_grads(rnn, head, canon.one_hot(S), S, canon.DELAY, truncate, loss_layer)
        grads = rnn.grads() + head.grads()
        if first is None:
            first = (loss, float(sum(g.sum() for g in grads)))
        clip_fn(grads, canon.CLIP)
        opt.step(rnn.params() + head.params(), grads)
    return canon.echo_accuracy(rnn, head, canon.DELAY, seed=seed), loss, first

mine = {k: train_echo_with(MyRNN, my_clip_grad_norm, truncate=k) for k in (0, 1)}
lib = {k: train_echo_with(ng.RNN, ng.clip_grad_norm, truncate=k) for k in (0, 1)}
for k, name in ((0, 'BPTT '), (1, 'Elman')):
    print(f'{name} twin with YOUR pieces: test accuracy {mine[k][0]:.4f}, final batch loss {mine[k][1]:.5f}')
assert mine[0][0] == 1.0, 'with full BPTT the network should echo perfectly'
assert mine[1][0] < 0.6, 'with the gradient cut after one step it should not learn'
compare_to_canonical([mine[0][2][0], mine[0][2][1], mine[1][2][1], mine[0][0], mine[1][0]],
                     [lib[0][2][0], lib[0][2][1], lib[1][2][1], canon.train_echo(truncate=0)['acc'], lib[1][0]],
                     labels=['loss0', 'grad0', 'grad0E', 'accBPTT', 'accElm'])
print('OK — your recurrence, your backpropagation through time and your clipping learn to remember; '
      'cut the gradient and they cannot.')